Language: **English (default)** | [Polski](lab.ipynb)

# Lab 13 -- implementing and analyzing DDPM from scratch (two moons)

Companion material to [`README.md`](README.md). Dataset: two-moons -- a
different problem than the lecture (there: a mixture of Gaussians on a
circle).

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F  # noqa: F401 - used in the TODO below

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")


def sample_two_moons(n, noise=0.08, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    n1 = n // 2
    n2 = n - n1
    theta1 = torch.rand(n1) * math.pi
    moon1 = torch.stack([torch.cos(theta1), torch.sin(theta1)], dim=1)
    theta2 = torch.rand(n2) * math.pi
    moon2 = torch.stack([1 - torch.cos(theta2), 1 - torch.sin(theta2) - 0.5], dim=1)
    X = torch.cat([moon1, moon2], dim=0)
    return (X + noise * torch.randn(X.shape[0], 2)) * 3.0


X_train = sample_two_moons(2000, seed=0)
print(f"X_train: {X_train.shape}, mean={X_train.mean(0)}, std={X_train.std(0)}")

## 1. Visualizing the data -- your task

Draw a scatter plot of `X_train`. In a comment/markdown: how does this
shape differ from the Gaussian mixture from the lecture, and why couldn't
a single Gaussian distribution describe it well?

In [ ]:
# TODO: plt.scatter(...) for X_train, plt.axis("equal"), plt.show()
raise NotImplementedError("TODO: data visualization")

## 2. The schedule β and computing ᾱ_t -- your task

In [ ]:
T = 500
BETA_START, BETA_END = 1e-4, 0.02

# TODO:
# betas = torch.linspace(BETA_START, BETA_END, T)
# alphas = 1 - betas
# alphas_bar = torch.cumprod(alphas, dim=0)
# print abar at t=1, T//2, T -- is abar_T close enough to 0?
raise NotImplementedError("TODO: schedule and alphas_bar")

## 3. The closed-form forward diffusion -- your task

Implement `q_sample` and verify it statistically (mean/variance over many
samples of `eps`) against a step-by-step simulation -- **do not** expect
sample-for-sample identity.

In [ ]:
# TODO:
# def q_sample(x0, t, eps, abar):
#     ...
#
# VERIFICATION (required): for a fixed x0 and t, compare mean/var between
# q_sample (N samples of eps) and the step-by-step simulation (N independent chains)
raise NotImplementedError("TODO: q_sample + statistical verification")

## 4. The noise-prediction network and time conditioning -- your task

`sinusoidal_embedding` is provided. `NoisePredictor.forward` must use it --
concatenate it with `x` before the linear layers.

In [ ]:
def sinusoidal_embedding(t, dim=16):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half).float() / half)
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

In [ ]:
# TODO:
# class NoisePredictor(nn.Module):
#     def __init__(self, time_emb_dim=16, hidden=128):
#         ...  # e.g. Linear(2+time_emb_dim,h)->ReLU->Linear(h,h)->ReLU->Linear(h,2)
#     def forward(self, x, t):
#         ...  # MUST use sinusoidal_embedding(t, ...) and concatenate with x
#
# Comment: why isn't x_t alone enough for the model without knowing t?
raise NotImplementedError("TODO: network + time conditioning")

## 5. The objective and the training loop -- your task

The loop skeleton (optimizer, iterating over epochs/batches) is provided
in `train_ddpm` -- your task is **inside** the loop (the training step).

In [ ]:
def train_ddpm(model, X, T_steps, abar, epochs=300, batch_size=128, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)  # noqa: F841 - used in the TODO below
    n = X.shape[0]
    losses = []
    t0 = time.time()
    for epoch in range(epochs):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i : i + batch_size]
            x0_batch = X[idx]
            b = x0_batch.shape[0]  # noqa: F841 - used in the TODO below
            # TODO:
            # 1. t = torch.randint(0, T_steps, (b,))
            # 2. eps = torch.randn(b, 2)
            # 3. x_t = q_sample(x0_batch, t, eps, abar)
            # 4. eps_pred = model(x_t, t)
            # 5. loss = F.mse_loss(eps_pred, eps)
            # 6. opt.zero_grad(); loss.backward(); opt.step()
            # 7. epoch_loss += loss.item() * b
            raise NotImplementedError("TODO: DDPM training step")
        losses.append(epoch_loss / n)
    return time.time() - t0, losses

In [ ]:
model = NoisePredictor()  # noqa: F821 - defined in section 4
train_time, losses = train_ddpm(model, X_train, T, alphas_bar)  # noqa: F821
print(f"train_time: {train_time:.2f}s")

plt.figure(figsize=(5, 3.5))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("L_simple")
plt.title("Training loss (monitoring)")
plt.tight_layout()
plt.show()

## 6. Reverse sampling -- your task

In [ ]:
def ddpm_reverse_step(model, x, t_idx, alphas, betas, alphas_bar):
    t_batch = torch.full((x.shape[0],), t_idx, dtype=torch.long)
    eps_pred = model(x, t_batch)  # noqa: F841 - used in the TODO below
    alpha_t, beta_t, abar_t = alphas[t_idx], betas[t_idx], alphas_bar[t_idx]  # noqa: F841
    # TODO:
    # mu = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - abar_t)) * eps_pred)
    # if t_idx > 0: return mu + torch.sqrt(beta_t) * torch.randn_like(x)
    # else: return mu  # LAST step: without added noise
    raise NotImplementedError("TODO: DDPM reverse step")


@torch.no_grad()
def ddpm_sample(model, n_samples, T_steps, alphas, betas, alphas_bar, seed=0):
    torch.manual_seed(seed)
    x = torch.randn(n_samples, 2)
    n_with_noise, n_without_noise = 0, 0
    for t_idx in reversed(range(T_steps)):
        x = ddpm_reverse_step(model, x, t_idx, alphas, betas, alphas_bar)
        if t_idx > 0:
            n_with_noise += 1
        else:
            n_without_noise += 1
    print(
        f"steps with added noise: {n_with_noise}, without noise: {n_without_noise} "
        f"(expected: {T_steps - 1} and 1)"
    )
    return x

In [ ]:
model.eval()
t0 = time.time()
samples = ddpm_sample(model, 1000, T, alphas, betas, alphas_bar, seed=42)  # noqa: F821
print(f"generation time: {time.time() - t0:.2f}s")

## 7. Generating new samples and comparing distributions -- your task

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(X_train[:, 0], X_train[:, 1], s=4, alpha=0.4)
axes[0].set_title("Training data")
axes[0].axis("equal")
axes[1].scatter(samples[:, 0], samples[:, 1], s=4, alpha=0.4, color="tab:orange")
axes[1].set_title("Generated samples (from pure noise, NOT reconstructions)")
axes[1].axis("equal")
plt.tight_layout()
plt.show()

In [ ]:
def rbf_mmd2(X, Y, sigma=None):
    XX, YY, XY = torch.cdist(X, X) ** 2, torch.cdist(Y, Y) ** 2, torch.cdist(X, Y) ** 2
    if sigma is None:
        combined = torch.cat([XX.flatten(), YY.flatten(), XY.flatten()])
        sigma = combined[combined > 0].median().sqrt().item()
    gamma = 1.0 / (2 * sigma**2)
    Kxx, Kyy, Kxy = torch.exp(-gamma * XX), torch.exp(-gamma * YY), torch.exp(-gamma * XY)
    nx, ny = X.shape[0], Y.shape[0]
    sum_xx = (Kxx.sum() - Kxx.diag().sum()) / (nx * (nx - 1))
    sum_yy = (Kyy.sum() - Kyy.diag().sum()) / (ny * (ny - 1))
    sum_xy = Kxy.sum() / (nx * ny)
    return (sum_xx + sum_yy - 2 * sum_xy).item(), sigma


def mean_nn_distance(samples, reference):
    return torch.cdist(samples, reference).min(dim=1).values.mean().item()

In [ ]:
# VERIFY MMD on a known case -- BEFORE using it on generated data
torch.manual_seed(1)
A = torch.randn(300, 2)
B_same = torch.randn(300, 2)
B_shifted = torch.randn(300, 2) + torch.tensor([5.0, 5.0])
mmd_same, sigma_ref = rbf_mmd2(A, B_same)
mmd_diff, _ = rbf_mmd2(A, B_shifted, sigma=sigma_ref)
print(f"control MMD^2: same distribution={mmd_same:.5f}, shifted={mmd_diff:.5f}")
assert mmd_same < 0.01
assert mmd_diff > mmd_same * 10

# TODO:
# X_test = sample_two_moons(1000, seed=123)  # INDEPENDENT test set
# mmd_gen, sigma_data = rbf_mmd2(X_test, samples)
# nn_gen = mean_nn_distance(samples, X_test)
# print both metrics
raise NotImplementedError("TODO: evaluating generation on an independent test set")

## 8. Checking a single configuration change -- your task

Choose **one** change (number of sampling steps with correct respacing,
the schedule, or the number of training epochs) and measure the effect
with the same metric.

In [ ]:
# TODO: choose ONE change, measure it with the same metric (mmd_gen or nn_gen from section 7)
raise NotImplementedError("TODO: one configuration change")

## 9. A baseline -- your task

A single fitted Gaussian distribution. Do not assume in advance which
model will win on every metric.

In [ ]:
# TODO:
# mu_fit = X_train.mean(0)
# cov_fit = torch.cov(X_train.T)
# baseline_dist = torch.distributions.MultivariateNormal(mu_fit, cov_fit)
# baseline_samples = baseline_dist.sample((1000,))
# compute mmd/nn-distance for baseline_samples against X_test, compare with section 7
raise NotImplementedError("TODO: Gaussian baseline")

## 10. Diagnostic task

The following code (provided, run it) implements `NoisePredictorColleague`
-- it accepts `t`, but **never uses** it in `forward`.

In [ ]:
class NoisePredictorColleague(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        return self.net(x)


model_colleague = NoisePredictorColleague()
_, losses_colleague = train_ddpm(model_colleague, X_train, T, alphas_bar)  # noqa: F821
samples_colleague = ddpm_sample(  # noqa: F821
    model_colleague,
    1000,
    T,
    alphas,  # noqa: F821
    betas,  # noqa: F821
    alphas_bar,  # noqa: F821
    seed=42,
)
print(f"final loss (colleague, no time conditioning): {losses_colleague[-1]:.4f}")
print(f"final loss (your model, section 5): {losses[-1]:.4f}")  # noqa: F821

In [ ]:
# TODO:
# 1. compute mmd/nn-distance for samples_colleague against X_test (section 7)
# 2. compare with your results from section 7
# 3. in a comment: explain WHY a model without t does worse, despite
#    still generating something (not crashing)
raise NotImplementedError("TODO: diagnosing the lack of time conditioning")

## 11. Conclusions -- your task

**1. Was ᾱ_T close enough to 0 for your schedule? What would have happened if it weren't?**

_(Your answer here.)_

**2. How well (specific metrics) does your model reproduce the two-moons distribution on the independent test set?**

_(Your answer here.)_

**3. How does it compare with the baseline? Did both metrics give the same answer?**

_(Your answer here.)_

**4. What effect did the configuration change from section 8 have?**

_(Your answer here.)_

**5. What error did the lack of time conditioning introduce, and what is the general rule?**

_(Your answer here.)_

**6. Are there regions of the data space the model doesn't generate well? How did you check this?**

_(Your answer here.)_